In [1]:
# !pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VjAxZSEJru3p0Pij8zYu")
project = rf.workspace("football-7ht3k").project("match-zcjsf")
version = project.version(1)
dataset = version.download("yolov8")
                


loading Roboflow workspace...
loading Roboflow project...


In [2]:
import yaml

# Lire le fichier généré par Roboflow
with open(f"{dataset.location}/data.yaml", 'r') as stream:
    data = yaml.safe_load(stream)
    print(data)
# print("Nombre de classes (nc) :", data['nc'])
# print("Noms des classes :", data['names'])

{'names': ['Ball', 'Keeper', 'Player', 'Ref'], 'nc': 4, 'roboflow': {'license': 'CC BY 4.0', 'project': 'match-zcjsf', 'url': 'https://universe.roboflow.com/football-7ht3k/match-zcjsf/dataset/1', 'version': 1, 'workspace': 'football-7ht3k'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [4]:
!pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# Exemple de ce que vous feriez juste après :
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Charge un modèle YOLOv8 natif
# model.train(data=f"{dataset.location}/data.yaml", epochs=50)  # Lance l'entraînement !

In [2]:
resultat  = model("../../../data/extracted_frames/data1/frame_1_490.jpg")
resultat[0].show()


image 1/1 D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\detection\fine-tunning_1\..\..\..\data\extracted_frames\data1\frame_1_490.jpg: 384x640 9 persons, 81.2ms
Speed: 4.5ms preprocess, 81.2ms inference, 5.8ms postprocess per image at shape (1, 3, 384, 640)


In [7]:
# 2. Lancer la validation exclusivement sur le jeu de test
metrics = model.val(split='test')

# 3. Afficher les résultats clés
print(f"mAP50-95 (Précision globale) : {metrics.box.map:.4f}")
print(f"mAP50 (Précision à seuil 0.5) : {metrics.box.map50:.4f}")
print(f"Précision par classe : {metrics.box.mp:.4f}")
print(f"Rappel (Recall) par classe : {metrics.box.mr:.4f}")

# Afficher les performances détaillées par classe
for i, name in enumerate(metrics.names.values()):
    print(f"--- Classe : {name} ---")
    print(f"  Précision (Precision) : {metrics.box.class_result(i)[0]:.4f}")
    print(f"  Rappel (Recall)       : {metrics.box.class_result(i)[1]:.4f}")
    print(f"  mAP50                 : {metrics.box.class_result(i)[2]:.4f}")
    print("-" * 25)

Ultralytics 8.4.60  Python-3.13.2 torch-2.12.0+cpu CPU (Intel Core i5-1035G1 1.00GHz)
val: Fast image access  (ping: 4.69.4 ms, read: 1.10.4 MB/s, size: 46.1 KB)
val: Scanning D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\match-1\test\labels... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 64.1it/s 0.2s
val: New cache created: D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\match-1\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.9s/it 2.9s
                   all         14        218      0.885      0.781      0.842      0.425
                  Ball         14         14      0.852      0.415      0.476      0.201
                Keeper          9          9      0.805      0.778      0.915      0.407
                Player         14        169      0.948     

In [8]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\train\weights\best.pt")
cap = cv2.VideoCapture("video.mp4")

# Récupérer les propriétés de la vidéo d'origine
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Créer le fichier de sortie
out = cv2.VideoWriter("resultat_annote.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    resultats = model(frame)
    frame_annotee = resultats[0].plot()

    out.write(frame_annotee)        # écrit la frame dans le fichier vidéo
    cv2.imshow("Detection", frame_annotee)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 320x640 1 Ball, 15 Players, 1 Ref, 186.8ms
Speed: 7.8ms preprocess, 186.8ms inference, 2.7ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Ball, 19 Players, 1 Ref, 156.8ms
Speed: 7.4ms preprocess, 156.8ms inference, 4.0ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Keeper, 14 Players, 124.2ms
Speed: 5.7ms preprocess, 124.2ms inference, 2.5ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Keeper, 13 Players, 1 Ref, 122.5ms
Speed: 4.8ms preprocess, 122.5ms inference, 2.5ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Keeper, 13 Players, 1 Ref, 119.9ms
Speed: 4.4ms preprocess, 119.9ms inference, 2.6ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Keeper, 14 Players, 1 Ref, 123.5ms
Speed: 4.8ms preprocess, 123.5ms inference, 2.5ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 Keeper, 17 Players, 125.0ms
Speed: 2.8ms preprocess, 125.0ms inference, 2.5ms postprocess per image at shape (1, 3,